# Fine-tuning on Kaggle (T4)

LoRA fine-tune of `Qwen2.5-Coder-1.5B-Instruct` on the CommitBench data produced by
`CommitBench_filter.ipynb`. Loss is computed on the commit message only (prompt/completion
format). The saved adapter is small (~70MB); download it and drop it into the local
`checkpoints/` folder for CLI inference, where it loads on top of a 4-bit base model.

**Setup on Kaggle:**
1. Settings -> Accelerator -> GPU T4 x2 (only one is used), Internet -> On
2. Add Data -> attach the dataset containing `train.jsonl` / `validation.jsonl` (found automatically under `/kaggle/input`)
3. Run with **Save Version -> Save & Run All (Commit)**, not interactively: training takes
   hours, and an interactive session dies if the browser tab idles or closes. The zip then
   appears in the version's Output tab.

In [ ]:
# --- Dependencies ---
# Kaggle's base image ships an old torchao that transformers rejects at import time.
# Pin it explicitly rather than blanket-upgrading (avoids pip also touching torch
# itself, which would risk breaking Kaggle's pre-tuned CUDA build).
%pip install -q -U transformers peft trl datasets accelerate "torchao>=0.16.0"

import os

# Single GPU: in a notebook, Trainer would otherwise wrap the model in nn.DataParallel
# across both T4s, which is flaky with LoRA + gradient checkpointing.
# Must be set before torch initializes CUDA — restart the session if torch was already imported.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import glob
import shutil

import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

assert torch.cuda.is_available(), "Enable a GPU accelerator (Settings -> Accelerator -> GPU T4 x2)"
print("GPUs visible:", torch.cuda.device_count(), torch.cuda.get_device_name(0))

In [ ]:
# --- Config, model, data, training ---

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"      # intermediate checkpoints (not downloaded)
OUTPUT_DIR = "/kaggle/working/commit-model-lora"    # final adapter only

SEQ_LEN = 768
LORA_R = 16
EPOCHS = 2.0          # avg example ~233 tokens -> roughly 1-1.5h per epoch on one T4
BATCH_SIZE = 2
GRAD_ACCUM = 8        # effective batch size 16
LEARNING_RATE = 2e-4
EVAL_SAMPLES = 500    # evaluating the full 9.6K validation set every 200 steps would add hours

PROMPT_TEMPLATE = "Write a Conventional Commits message for this diff.\n\n{diff}\n\nCommit message:"


def find_input_file(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not matches:
        raise FileNotFoundError(
            f"Couldn't find {filename} anywhere under /kaggle/input. "
            f"Check Add Data attached the dataset. Found under /kaggle/input: "
            f"{glob.glob('/kaggle/input/**/*', recursive=True)}"
        )
    return matches[0]


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

TEMPLATE_TOKENS = len(tokenizer(PROMPT_TEMPLATE.format(diff=""), add_special_tokens=False).input_ids)


def to_prompt_completion(ex):
    completion = " " + ex["message"] + tokenizer.eos_token
    completion_tokens = len(tokenizer(completion, add_special_tokens=False).input_ids)
    # Truncate the diff, not the message: otherwise right-truncation at SEQ_LEN would
    # cut off the commit message, which is the only part we train on.
    budget = max(SEQ_LEN - TEMPLATE_TOKENS - completion_tokens - 8, 0)
    diff_ids = tokenizer(ex["diff"], add_special_tokens=False).input_ids[:budget]
    return {
        "prompt": PROMPT_TEMPLATE.format(diff=tokenizer.decode(diff_ids)),
        "completion": completion,
    }


data_files = {
    "train": find_input_file("train.jsonl"),
    "validation": find_input_file("validation.jsonl"),
}
print("Using data files:", data_files)
dataset = load_dataset("json", data_files=data_files)
dataset = dataset.map(to_prompt_completion, remove_columns=["diff", "message"])

eval_dataset = dataset["validation"].shuffle(seed=42).select(
    range(min(EVAL_SAMPLES, len(dataset["validation"])))
)

# fp32 weights + fp16 autocast: T4 has no bf16 support, and pure-fp16 weights risk
# overflow with Qwen. 1.5B in fp32 is ~6GB, fine on a 16GB T4.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float32, device_map={"": 0}
)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_R * 2,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

training_args = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    max_length=SEQ_LEN,
    completion_only_loss=True,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    fp16=True,
    bf16=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved LoRA adapter to {OUTPUT_DIR}")


# --- Zip the adapter for download (final adapter only, no checkpoints) ---
shutil.make_archive("/kaggle/working/commit-model-lora", "zip", OUTPUT_DIR)
print("Download commit-model-lora.zip from the Kaggle output panel, then locally:")
print("  mkdir -p checkpoints && unzip commit-model-lora.zip -d checkpoints/commit-model-lora")